# 18 - Gold Data Quality Dashboard

Aggregates persisted valid Bronze records, quarantined errors, and duplicate-key metrics by source and date.

This notebook is deterministic and safe to rerun. It synchronizes
the complete business result with Delta `MERGE`, records an audit
run, and refreshes aggregate-to-source lineage.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`gold`")


def sync_gold_table(target_table, source_df, keys):
    """Synchronize a deterministic Gold snapshot with an idempotent Delta MERGE."""
    source_df = source_df.dropDuplicates(keys)
    target_df = spark.table(target_table)
    stale_keys = (
        target_df.select(*keys)
        .join(source_df.select(*keys), keys, "left_anti")
        .withColumn("_delete", F.lit(True))
    )
    merge_source = (
        source_df.withColumn("_delete", F.lit(False))
        .unionByName(stale_keys, allowMissingColumns=True)
    )
    condition = " AND ".join(
        f"target.`{column}` = source.`{column}`" for column in keys
    )
    assignments = {
        column: f"source.`{column}`" for column in source_df.columns
    }
    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(merge_source.alias("source"), condition)
        .whenMatchedDelete(condition="source._delete")
        .whenMatchedUpdate(
            condition=(
                "NOT source._delete "
                "AND NOT (target.record_hash <=> source.record_hash)"
            ),
            set=assignments,
        )
        .whenNotMatchedInsert(
            condition="NOT source._delete",
            values=assignments,
        )
        .execute()
    )


def replace_gold_lineage(target_table, lineage_df):
    """Keep the current direct source mappings for one aggregate Gold table."""
    lineage_target = DeltaTable.forName(
        spark, f"{catalog}.control.record_lineage"
    )
    lineage_target.delete(F.col("target_table") == F.lit(target_table))
    prepared = (
        lineage_df.select(
            F.lit(target_table).alias("target_table"),
            F.col("target_record_key").cast("string"),
            F.lit(pipeline_run_id).alias("pipeline_run_id"),
            "source_table",
            F.col("source_record_key").cast("string"),
            "source_file",
            "batch_id",
            F.col("source_ingest_ts").cast("timestamp"),
            F.current_timestamp().alias("recorded_ts"),
        )
        .dropDuplicates(
            [
                "target_table",
                "target_record_key",
                "source_table",
                "source_record_key",
                "source_file",
                "batch_id",
            ]
        )
    )
    if prepared.limit(1).count():
        prepared.write.format("delta").mode("append").saveAsTable(
            f"{catalog}.control.record_lineage"
        )


In [ ]:
from pyspark.sql.window import Window

pipeline_name = "gold_data_quality_dashboard"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
target_table = f"{catalog}.gold.gold_data_quality_dashboard"
bronze_sources = [
    ("clickstream_events", f"{catalog}.bronze.bronze_clickstream"),
    ("orders", f"{catalog}.bronze.bronze_orders"),
    ("customers", f"{catalog}.bronze.bronze_customers"),
]
quarantine_sources = [
    ("clickstream_events", f"{catalog}.quarantine.clickstream_errors"),
    ("orders", f"{catalog}.quarantine.orders_errors"),
    ("customers", f"{catalog}.quarantine.customers_errors"),
]
dq_table = f"{catalog}.control.data_quality_results"
source_tables = [table for _, table in bronze_sources + quarantine_sources] + [dq_table]

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`gold`.`gold_data_quality_dashboard` (
        quality_date DATE NOT NULL,
        source_name STRING NOT NULL,
        quality_status STRING NOT NULL,
        error_type STRING NOT NULL,
        record_count BIGINT NOT NULL,
        record_hash STRING NOT NULL,
        updated_ts TIMESTAMP NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    PARTITIONED BY (quality_date)
    TBLPROPERTIES (
        'comment' = 'Valid and classified error counts by source and date'
    )
    """
)

upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
    "RUNNING", source_tables, job_run_id,
)

try:
    valid_frames = []
    valid_lineage_frames = []
    for source_name, table_name in bronze_sources:
        bronze = spark.table(table_name)
        valid_frames.append(
            bronze.groupBy("load_date")
            .count()
            .select(
                F.col("load_date").alias("quality_date"),
                F.lit(source_name).alias("source_name"),
                F.lit("valid").alias("quality_status"),
                F.lit("valid").alias("error_type"),
                F.col("count").cast("long").alias("record_count"),
            )
        )
        valid_lineage_frames.append(
            bronze.select(
                F.concat_ws(
                    "|", F.col("load_date"), F.lit(source_name), F.lit("valid")
                ).alias("target_record_key"),
                F.lit(table_name).alias("source_table"),
                F.sha2(
                    F.concat_ws("||", "source_file", "raw_payload"), 256
                ).alias("source_record_key"),
                "source_file", "batch_id",
                F.col("ingest_ts").alias("source_ingest_ts"),
            )
        )

    error_frames = []
    error_lineage_frames = []
    for source_name, table_name in quarantine_sources:
        quarantine = spark.table(table_name)
        error_frames.append(
            quarantine.groupBy(
                F.coalesce("load_date", "quarantine_date").alias("quality_date"),
                "error_code",
            )
            .count()
            .select(
                "quality_date",
                F.lit(source_name).alias("source_name"),
                F.lit("error").alias("quality_status"),
                F.col("error_code").alias("error_type"),
                F.col("count").cast("long").alias("record_count"),
            )
        )
        error_lineage_frames.append(
            quarantine.select(
                F.concat_ws(
                    "|",
                    F.coalesce("load_date", "quarantine_date"),
                    F.lit(source_name),
                    F.col("error_code"),
                ).alias("target_record_key"),
                F.lit(table_name).alias("source_table"),
                F.col("error_id").alias("source_record_key"),
                "source_file", "batch_id",
                F.col("ingest_ts").alias("source_ingest_ts"),
            )
        )

    valid = valid_frames[0]
    for frame in valid_frames[1:]:
        valid = valid.unionByName(frame)
    errors = error_frames[0]
    for frame in error_frames[1:]:
        errors = errors.unionByName(frame)

    dq_metrics = spark.table(dq_table).filter(
        (F.col("quality_status") == "error")
        & (F.col("error_code") == "duplicate_key")
    )
    latest_metric = Window.partitionBy("source_name", "error_code").orderBy(
        F.col("recorded_ts").desc(), F.col("pipeline_run_id").desc()
    )
    latest_duplicate_rows = (
        dq_metrics.withColumn("_rank", F.row_number().over(latest_metric))
        .filter((F.col("_rank") == 1) & (F.col("record_count") > 0))
    )
    duplicate_metrics = (
        latest_duplicate_rows
        .select(
            "quality_date", "source_name",
            F.lit("error").alias("quality_status"),
            F.col("error_code").alias("error_type"),
            F.col("record_count").cast("long").alias("record_count"),
        )
    )

    business = (
        valid.unionByName(errors)
        .unionByName(duplicate_metrics)
        .groupBy("quality_date", "source_name", "quality_status", "error_type")
        .agg(F.sum("record_count").cast("long").alias("record_count"))
    )
    source = (
        business.withColumn(
            "record_hash",
            F.sha2(F.col("record_count").cast("string"), 256),
        )
        .withColumn("updated_ts", F.current_timestamp())
        .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
    )
    sync_gold_table(
        target_table,
        source,
        ["quality_date", "source_name", "quality_status", "error_type"],
    )

    valid_lineage = valid_lineage_frames[0]
    for frame in valid_lineage_frames[1:]:
        valid_lineage = valid_lineage.unionByName(frame)
    error_lineage = error_lineage_frames[0]
    for frame in error_lineage_frames[1:]:
        error_lineage = error_lineage.unionByName(frame)
    duplicate_lineage = latest_duplicate_rows.select(
        F.concat_ws(
            "|", "quality_date", "source_name", "error_code"
        ).alias("target_record_key"),
        F.lit(dq_table).alias("source_table"),
        F.concat_ws("|", "pipeline_run_id", "error_code")
        .alias("source_record_key"),
        F.lit(None).cast("string").alias("source_file"),
        F.col("pipeline_run_id").alias("batch_id"),
        F.col("recorded_ts").alias("source_ingest_ts"),
    )
    replace_gold_lineage(
        target_table,
        valid_lineage.unionByName(error_lineage).unionByName(duplicate_lineage),
    )

    rows_read = sum(spark.table(table).count() for table in source_tables)
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
        "SUCCESS", source_tables, job_run_id,
        end_ts=datetime.now(timezone.utc), rows_read=rows_read,
        rows_written=source.count(), rows_quarantined=0,
    )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
        "FAILED", source_tables, job_run_id,
        end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
    )
    raise

spark.table(target_table).orderBy("quality_date", "source_name", "error_type").display()
